# Herb-Hermes 语音研究驾驶舱（GPU 部署）

在带 GPU 的 Colab / 服务器上，把 **FireRedASR2-AED（语音转文字）** 与 **CosyVoice3-0.5B（文字转语音）** 接入 Herb-Hermes API，实现「语音提问本草 / 方剂 → 系统检索 → 语音播报」的完整闭环。

本 notebook 改编自官方 FireRedASR2S / CosyVoice3 复现脚本，额外启动 Herb-Hermes 服务并通过环境变量挂载两个语音后端。

> 无 GPU 时，前端会自动回退到浏览器 Web Speech API（识别 + 朗读），核心检索功能不受影响。

In [ ]:
!nvidia-smi
!python -V

## 1. 获取 Herb-Hermes 与依赖

In [ ]:
%cd /content
# 替换为你的仓库地址；或将本仓库上传到 /content/Herb-Hermes
![ -d Herb-Hermes ] || git clone https://github.com/pariskang/Herb-Hermes.git
%cd /content/Herb-Hermes
!pip install -q fastapi uvicorn
# 构建知识库（首次约 1-2 分钟）。若已随仓库携带 data/processed 可跳过。
!python -m herb_hermes.index_build

## 2. 安装并下载 FireRedASR2-AED（ASR）

In [ ]:
%cd /content
![ -d FireRedASR2S ] || git clone https://github.com/FireRedTeam/FireRedASR2S.git
%cd /content/FireRedASR2S
# 官方 requirements 固定了 torch==2.1.0+cu118，Colab 上需放宽
!sed -i 's/torch==2.1.0+cu118//g' requirements.txt
!pip install -q -r requirements.txt || true
!apt-get install -y -qq ffmpeg

from pathlib import Path
from huggingface_hub import snapshot_download
ASR_DIR = Path('/content/FireRedASR2S/pretrained_models/FireRedASR2-AED')
ASR_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id='FireRedTeam/FireRedASR2-AED', local_dir=str(ASR_DIR), resume_download=True)
print('ASR model at', ASR_DIR)

## 3. 安装并下载 CosyVoice3-0.5B（TTS）

In [ ]:
%cd /content
!apt-get install -y -qq sox libsox-dev ffmpeg git-lfs
![ -d CosyVoice ] || git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd /content/CosyVoice
!git submodule update --init --recursive
!pip install -q torch==2.3.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121 || true
!grep -vE '^(torch==|torchaudio==|deepspeed==|tensorrt-cu12)' requirements.txt > /tmp/cv_req.txt && pip install -q -r /tmp/cv_req.txt || true

from pathlib import Path
from huggingface_hub import snapshot_download
TTS_DIR = Path('/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B')
TTS_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id='FunAudioLLM/Fun-CosyVoice3-0.5B-2512', local_dir=str(TTS_DIR), resume_download=True)
print('TTS model at', TTS_DIR)

## 4. 配置后端环境变量并启动 Herb-Hermes 服务

In [ ]:
import os
os.environ['HERB_HERMES_ASR_MODEL_DIR'] = '/content/FireRedASR2S/pretrained_models/FireRedASR2-AED'
os.environ['HERB_HERMES_TTS_MODEL_DIR'] = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
os.environ['HERB_HERMES_COSYVOICE_REPO'] = '/content/CosyVoice'
# 朗读音色参考音频（用官方示例，或换成你有授权的音频）
os.environ['HERB_HERMES_TTS_PROMPT_WAV'] = '/content/CosyVoice/asset/zero_shot_prompt.wav'
# 让 fireredasr2s 与 cosyvoice 可被 import
os.environ['PYTHONPATH'] = '/content/FireRedASR2S:/content/CosyVoice:/content/CosyVoice/third_party/Matcha-TTS:' + os.environ.get('PYTHONPATH','')
print('Backends configured.')

In [ ]:
%cd /content/Herb-Hermes
# 用 pyngrok 暴露公网地址（或在本地直接访问 8000 端口）
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, time, os
proc = subprocess.Popen(['uvicorn', 'herb_hermes.api.server:app', '--host', '0.0.0.0', '--port', '8000'],
                        env=os.environ.copy())
time.sleep(8)
url = ngrok.connect(8000).public_url
print('研究驾驶舱：', url + '/app/')
print('API 文档：', url + '/docs')
print('语音后端状态：', url + '/voice/status')

## 5. 自检

打开上面的「研究驾驶舱」链接，点击右上角 🎤 用普通话说「黄芪」或「桂枝汤」，系统会语音识别→自动检索→展示结果；在「科研假设」页点 🔊 朗读会用 CosyVoice3 播报。

也可直接调用 API：

In [ ]:
import requests
print(requests.get(url + '/voice/status').json())
# ASR：上传一段音频字节
# data = open('test.wav','rb').read()
# print(requests.post(url+'/voice/asr', data=data).json())
# TTS：合成并保存
# r = requests.post(url+'/voice/tts', json={'text':'黄芪补气固表，托毒生肌。'})
# open('out.wav','wb').write(r.content)